# 01 - Plain WGAN-GP on MNIST

**Where this sits:** **`WGAN-GP`** -> RWGAN -> M-RWGAN. This is the baseline: a
Wasserstein GAN with gradient penalty and **no reward signal**. The generator is
pulled only toward "look like real MNIST", so it should learn the *whole* data
distribution - all ten digit classes, in roughly natural proportions.

Notebooks 02 and 03 keep this exact WGAN-GP core and add frozen "reward" CNNs
that pull the generator toward chosen digits.

**Self-contained:** every model / training / plotting block is inline; nothing is
imported from the repo's `mrwgan` package. The three notebooks intentionally
repeat their shared blocks so each reads top-to-bottom on its own.

## What this notebook does
1. Load MNIST, scale to `[-1, 1]`, shape `(28, 28, 1)`.
2. Build a CNN generator and a CNN critic (no BatchNorm in the critic).
3. Train with the WGAN-GP recipe: `n_critic = 5`, gradient-penalty weight 10,
   `Adam(2e-4, 0.5, 0.9)` for both networks, gradient penalty via `tf.GradientTape`.
4. Save a fixed-noise 8x8 sample grid to `output/01_wgan_gp_samples.png`.

## Imports

In [ ]:
import os
import time

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__, "| executing eagerly:", tf.executing_eagerly())

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Configuration

In [ ]:
NB_TAG = "01_wgan_gp"

EPOCHS = 40            # set to 2-3 for a quick smoke run
N_EVAL_SAMPLES = 2000  # generated images for the digit-class histogram (unused in nb1; kept for parity)

BATCH_SIZE = 128
LATENT_DIM = 100
N_CRITIC = 5           # critic updates per generator update (WGAN-GP)
GP_WEIGHT = 10.0       # gradient-penalty coefficient

## Data -- MNIST scaled to [-1, 1]

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


def preprocess(images):
    """uint8 [0, 255] -> float32 [-1, 1], shape (N, 28, 28, 1)."""
    images = images.astype("float32")
    images = (images - 127.5) / 127.5
    return images[..., np.newaxis]


x_train_img = preprocess(x_train)
x_test_img = preprocess(x_test)

train_ds = (
    tf.data.Dataset.from_tensor_slices(x_train_img)
    .shuffle(60_000, seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)
STEPS_PER_EPOCH = x_train_img.shape[0] // BATCH_SIZE
ds_iter = iter(train_ds)
print("train images:", x_train_img.shape, "| range:", float(x_train_img.min()), float(x_train_img.max()))
print("steps per epoch:", STEPS_PER_EPOCH)

## Models -- CNN generator + CNN critic

In [ ]:
def make_generator():
    """z(100) -> 7x7x256 -> 14x14x128 -> 28x28x64 -> 28x28x1 (tanh)."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(LATENT_DIM,)),
            tf.keras.layers.Dense(7 * 7 * 256, use_bias=False),
            tf.keras.layers.Reshape((7, 7, 256)),
            tf.keras.layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # -> 14x14
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),   # -> 28x28
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Conv2D(1, 7, padding="same", activation="tanh"),
        ],
        name="generator",
    )


def make_critic():
    """CNN critic. No BatchNorm: the WGAN-GP gradient penalty is a per-sample
    constraint and BatchNorm mixes statistics across the batch. LayerNorm is safe."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(64, 5, strides=2, padding="same"),   # -> 14x14
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Conv2D(128, 5, strides=2, padding="same"),  # -> 7x7
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.LeakyReLU(0.2),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(1),  # linear score
        ],
        name="critic",
    )


generator = make_generator()
critic = make_critic()
generator.summary()
critic.summary()

## WGAN-GP core -- optimizers, gradient penalty, critic step

In [ ]:
g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5, beta_2=0.9)
d_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5, beta_2=0.9)


def gradient_penalty(real, fake):
    """E[(||grad_x critic(x_hat)||_2 - 1)^2], x_hat = eps*real + (1-eps)*fake, eps ~ U[0, 1]."""
    batch = tf.shape(real)[0]
    eps = tf.random.uniform([batch, 1, 1, 1], 0.0, 1.0)
    x_hat = eps * real + (1.0 - eps) * fake
    with tf.GradientTape() as tape:
        tape.watch(x_hat)
        d_hat = critic(x_hat, training=True)
    grads = tape.gradient(d_hat, x_hat)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
    return tf.reduce_mean((norm - 1.0) ** 2)


@tf.function  # autograph-traced eager code -- NOT tf.compat.v1 graph mode
def train_critic_step(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT_DIM])
    fake = generator(z, training=True)
    with tf.GradientTape() as tape:
        d_real = critic(real, training=True)
        d_fake = critic(fake, training=True)
        wasserstein = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real)
        gp = gradient_penalty(real, fake)
        d_loss = wasserstein + GP_WEIGHT * gp
    grads = tape.gradient(d_loss, critic.trainable_variables)
    d_optimizer.apply_gradients(zip(grads, critic.trainable_variables))
    return d_loss

## Sampling / plotting helpers

In [ ]:
fixed_noise = tf.random.normal([64, LATENT_DIM], seed=SEED)


def save_sample_grid(path, title, show=True):
    imgs = generator(fixed_noise, training=False).numpy()
    imgs = np.clip((imgs + 1.0) / 2.0, 0.0, 1.0)
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i, :, :, 0], cmap="gray")
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    if show:
        plt.show()
    plt.close(fig)


def generate_images(n, batch=500):
    out = []
    for i in range(0, n, batch):
        z = tf.random.normal([min(batch, n - i), LATENT_DIM])
        out.append(generator(z, training=False).numpy())
    return np.concatenate(out, axis=0)

## Generator step and training loop

Plain WGAN-GP: the generator loss is just `-E[critic(G(z))]`. There is no reward
term and therefore no lambda to anneal (notebooks 02 and 03 add both).

In [ ]:
@tf.function
def train_generator_step():
    z = tf.random.normal([BATCH_SIZE, LATENT_DIM])
    with tf.GradientTape() as tape:
        fake = generator(z, training=True)
        g_loss = -tf.reduce_mean(critic(fake, training=True))  # plain WGAN-GP generator loss
    grads = tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
    return g_loss


history = {"d_loss": [], "g_loss": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    d_hist, g_hist = [], []
    for _ in range(STEPS_PER_EPOCH):
        for _ in range(N_CRITIC):
            d_hist.append(float(train_critic_step(next(ds_iter))))
        g_hist.append(float(train_generator_step()))
    history["d_loss"].append(float(np.mean(d_hist)))
    history["g_loss"].append(float(np.mean(g_hist)))
    print(f"epoch {epoch:3d}/{EPOCHS} | d_loss {np.mean(d_hist):+.3f} | g_loss {np.mean(g_hist):+.3f} | {time.time() - t0:.1f}s")
    if epoch % 10 == 0 or epoch == EPOCHS:
        save_sample_grid(
            os.path.join(OUTPUT_DIR, f"{NB_TAG}_samples_epoch{epoch:03d}.png"),
            f"{NB_TAG} - epoch {epoch}",
            show=False,
        )

save_sample_grid(os.path.join(OUTPUT_DIR, f"{NB_TAG}_samples.png"), f"{NB_TAG} - final ({EPOCHS} epochs)")

## Training losses

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["d_loss"], label="critic loss")
ax.plot(history["g_loss"], label="generator loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
ax.set_title(f"{NB_TAG} - training losses")
fig.tight_layout()
plt.show()

## Result

With no reward term the generator is pulled only toward "look like real MNIST".
The fixed-noise grid in `output/01_wgan_gp_samples.png` should show a **varied
mix of all ten digits** in roughly natural proportions, with no class singled
out. Legibility improves with more epochs (`EPOCHS = 40` is a reasonable point;
2-3 is only a smoke test).

This is the reference behaviour that notebook 02 (one reward) and notebook 03
(two rewards) deliberately distort.